In [ ]:
!ls -la /content
!ls -la /content/drive
!rm -rf /content/drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)



In [ ]:
import os, sys, time

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("7006SCN_HMDA2023_Task2")
    .master("local[*]")
    .config("spark.driver.memory", "10g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

DATA    = "/content/drive/MyDrive/7006SCN/data/raw"
OUTPUT  = "/content/drive/MyDrive/7006SCN/outputs"
MODELS  = "/content/drive/MyDrive/7006SCN/models"

for p in [OUTPUT, MODELS]:
    os.makedirs(p, exist_ok=True)

CSV_PATH = f"{DATA}/My_Data.csv"

In [ ]:
# ── Commit 1: Ingestion + Repartition + Partition Count (CW mandatory) ─────────

from pyspark import StorageLevel
import time

start = time.time()

print("="*50)
print("HMDA 2023 DATA INGESTION")
print("="*50)

# Load dataset
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH)
)

# Cache immediately after loading
df.persist(StorageLevel.MEMORY_AND_DISK)

# Materialise cache
rows = df.count()
cols = len(df.columns)

load_time = time.time() - start

print(f"Rows               : {rows:,}")
print(f"Columns            : {cols}")
print(f"Load Time          : {load_time:.2f} seconds")
print()

print(f"Default partitions : {df.rdd.getNumPartitions()}")

# Repartition for balanced distributed processing
df = df.repartition(200)

print(f"After repartition  : {df.rdd.getNumPartitions()}")
print()

print(f"Storage Level      : {df.storageLevel}")

print()

print("Spark Version      :", spark.version)
print("Spark UI           :", spark.sparkContext.uiWebUrl)

print()

print("<-- Screenshot this output for Task 2 and Task 4 reports")

In [ ]:
print(f"Estimated rows/partition : {rows//df.rdd.getNumPartitions():,}")

Estimated rows/partition : 57,820


In [ ]:
# ── Commit 2: Data Quality Assessment + Cleaning (CW mandatory) ────────────────
# Commit 2: feat(task2): perform data quality assessment and preprocessing

# Dataset Overview
import time
from pyspark.sql import functions as F

start = time.time()

print("="*80)
print("HMDA 2023 DATA QUALITY ASSESSMENT")
print("="*80)

print(f"Rows                 : {df.count():,}")
print(f"Columns              : {len(df.columns)}")
print(f"Partitions           : {df.rdd.getNumPartitions()}")
print()

# Data Types Summary
print("="*50)
print("COLUMN DATA TYPES")
print("="*50)

type_summary = (
    df.dtypes
)

for name, dtype in type_summary:
    print(f"{name:<40} {dtype}")
# Missing Value Analysis
print("\n" + "="*50)
print("MISSING VALUE ANALYSIS")
print("="*50)

rows = df.count()

missing = []

for c in df.columns:
    col_type = dict(df.dtypes)[c]

    if col_type == "string":
        # For string columns, check for nulls or empty strings
        null_condition = F.col(c).isNull() | (F.col(c) == "")
    elif col_type in ["double", "float"]:
        # For float/double columns, check for nulls or NaN
        null_condition = F.col(c).isNull() | F.isnan(c)
    else:
        # For other numeric types (int, bigint, etc.), only check for nulls.
        # Comparing an int/bigint column to "" will cause the casting error.
        null_condition = F.col(c).isNull()

    nulls = df.filter(null_condition).count()

    missing.append((
        c,
        nulls,
        round((nulls/rows)*100,2)
    ))

missing_df = (
    spark.createDataFrame(
        missing,
        ["Feature","Missing Count","Missing %"]
    )
    .orderBy(F.desc("Missing %"))
)

missing_df.show(99, truncate=False)
# Duplicate Analysis

print("\n" + "="*50)
print("DUPLICATE ANALYSIS")
print("="*50)

duplicate_rows = rows - df.dropDuplicates().count()

print(f"Duplicate Records : {duplicate_rows:,}")
print(f"Duplicate %       : {(duplicate_rows/rows)*100:.4f}%")
# Target Distribution
print("\n" + "="*50)
print("TARGET DISTRIBUTION")
print("="*50)

(
df.groupBy("action_taken")
.count()
.orderBy("action_taken")
.show(truncate=False)
)
# Data Leakage Assessment
print("\n" + "="*50)
print("DATA LEAKAGE ASSESSMENT")
print("="*50)

potential_leakage = [

    "action_taken",

    "denial_reason_1",

    "denial_reason_2",

    "denial_reason_3",

    "denial_reason_4"

]

print("Potential leakage variables:")

for col in potential_leakage:
    if col in df.columns:
        print(f"✓ {col}")

# Numeric Summary
print("\n" + "="*50)
print("NUMERIC FEATURE SUMMARY")
print("="*50)

numeric = [

c for c,t in df.dtypes

if t in ("int","bigint","double","float","long")

]

df.select(numeric).describe().show()

print("""
================ PREPROCESSING STRATEGY ================

Missing Values
--------------
• Numeric features -> Median Imputation
• Categorical features -> 'Unknown'

Duplicates
----------
• Remove duplicate applications if present

Outliers
--------
• Detect using IQR
• Retain unless clearly invalid

Encoding
--------
• StringIndexer
• OneHotEncoder

Scaling
-------
• StandardScaler (for Logistic Regression)

========================================================
""")


In [19]:
# Processing Time
end = time.time()

print("="*80)
print(f"Commit 2 Execution Time : {(end-start):.2f} seconds")
print("="*80)

Commit 2 Execution Time : 2759.39 seconds
